# 04：文本 LoRA 训练与固定测试对照

按 **步骤 1 → 2 → 3** 执行，选择 `Python (Text LoRA - Qwen)` kernel。

agent 已编写 30 条训练数据与 6 条独立验证数据，复用你批准的 5 条答案。你的工作是监督样本、查看对照，并决定是否调整写法或增加数据。

默认复用本轮已经成功完成且配置匹配的结果，避免重复占用 GPU。将 `REUSE_COMPLETED_RUNS=False` 可重新实验；每次创建新目录，smoke 和 full 都从原模型重新初始化 LoRA。模型服务仍在运行时，先停止你自己的模型服务，再运行 GPU Cell。

这 36 条是小规模教学数据，用于验证流程；loss 和 10 条固定测试不能证明广泛的效果提升。

In [1]:
import html
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import HTML, Markdown, display

project_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                     for candidate in (parent, parent / "text-lora")
                     if (candidate / "scripts/check_environment.py").is_file()), None)
if project_root is None:
    raise RuntimeError("请从 text-lora/notebooks 打开此 Notebook")
if str(project_root / "scripts") not in sys.path:
    sys.path.insert(0, str(project_root / "scripts"))
from check_environment import check_python
from prepare_training_data import canonical_hash, load_training_config, validate_corpus
from run_inference import load_inference_config, prompts_fingerprint
from train_lora import training_kwargs, verify_adapter
from compare_inference import load_run

check_python()
REUSE_COMPLETED_RUNS = True
receipt_path = project_root / "outputs/latest_teaching_run.json"
receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
BASE_DIR = project_root / "outputs/base-20261008T012651_918242Z"


def new_run(prefix):
    return project_root / "outputs" / (prefix + "-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ"))


def run_script(script, *arguments):
    log_path = new_run("logs/" + Path(script).stem).with_suffix(".log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", str(project_root / "scripts" / script), *map(str, arguments)]
    print("运行：", script, *arguments)
    print("完整日志：", log_path)
    process = subprocess.Popen(command, cwd=project_root, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log_path.open("x", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                if any(tag in line for tag in ("[PASS]", "{'loss':", "{'eval_loss':", "推理完成")):
                    print(line.strip(), flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        print(log_path.read_text(encoding="utf-8")[-12000:])
        raise RuntimeError(f"{script} 失败，exit={returncode}；agent 根据完整日志修复")
    return log_path


def save_receipt():
    receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def reusable_training(key, mode):
    if not REUSE_COMPLETED_RUNS or key not in receipt:
        return None
    path = project_root / receipt[key]
    metadata = json.loads((path / "run_metadata.json").read_text())
    samples, prompts = validate_corpus()
    expected = training_kwargs(mode, path)
    config_matches = all(metadata["kwargs"].get(k) == v for k, v in expected.items()
                         if k not in {"dataset", "val_dataset"})
    if (metadata["status"] != "complete" or metadata["mode"] != mode or not config_matches
            or metadata["corpus_sha256"] != canonical_hash(samples)
            or metadata["prompts_sha256"] != prompts_fingerprint(prompts)):
        print("已有结果与当前数据/参数不匹配，将新建运行。")
        return None
    verify_adapter(metadata["selected_adapter"]["path"])
    print("复用已完成且数据/配置匹配的训练：", path)
    return path


def show_training(path):
    metadata = json.loads((path / "run_metadata.json").read_text())
    assert metadata["status"] == "complete"
    print("优化步数：", metadata["result"]["global_step"])
    print("峰值 allocated/reserved GiB：", metadata["peak_allocated_gib"], metadata["peak_reserved_gib"])
    print("选中的 adapter：", metadata["selected_adapter"]["path"])
    rows = ["| step | train loss | val loss |", "| --- | --- | --- |"]
    for entry in metadata["result"]["log_history"]:
        if "loss" in entry or "eval_loss" in entry:
            rows.append(f"| {entry['step']} | {entry.get('loss', '')} | {entry.get('eval_loss', '')} |")
    display(Markdown("\n".join(rows)))


Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3


## 步骤 1：监督数据并确认检查结果

直接运行下列 Cell。`teaching_samples.json` 保存完整样本与来源，导出脚本自动生成训练/验证 JSONL。检查包括：批准的 pilot 保持原样、故事来源分离、重复/近似测试输入检测、标签格式和真实训练模板 token 长度。近似文本检测是启发式；agent 还按故事来源和逐句事实做了复核。

展开样本可看原文、答案和核对项。新增答案的用户审阅状态保持待决策，程序不会代填 `HUMAN_REVIEWED`。

In [4]:
run_script("prepare_training_data.py", "--check-tokens")
samples, prompts = validate_corpus()
checks = json.loads((project_root / "datasets/processed/teaching-v1/checks.json").read_text())
print("训练/验证条数：", sum(s["split"] == "train" for s in samples), sum(s["split"] == "val" for s in samples))
print("完整模板最大长度：", max(x["total_tokens"] for x in checks["lengths"]))
print("训练参数：", load_training_config())
preview = []
for sample in samples:
    preview.append("<details><summary>" + html.escape(f"{sample['split']} / {sample['id']} / {sample['category']}")
                   + "</summary><p>原文：</p><pre>" + html.escape(sample["source_text"])
                   + "</pre><p>答案：</p><pre>" + html.escape(sample["messages"][-1]["content"])
                   + "</pre><p>检查项：" + html.escape("；".join(sample["review_criteria"])) + "</p></details>")
display(HTML("".join(preview)))
run_script("train_lora.py", "--mode", "smoke", "--dry-run")


运行： prepare_training_data.py --check-tokens
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/prepare_training_data-20261008T132438_770643Z.log
[PASS] 30 条训练 + 6 条验证；pilot 保留；来源分离、格式和测试隔离检查通过。
训练/验证条数： 30 6
完整模板最大长度： 232
训练参数： {'max_length': 512, 'learning_rate': 0.0001, 'num_train_epochs': 3, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1, 'gradient_accumulation_steps': 8, 'lora_rank': 8, 'lora_alpha': 32, 'lora_dropout': 0.05, 'seed': 42, 'data_seed': 42, 'smoke_steps': 3, 'loss_scale': 'default+ignore_empty_think'}


运行： train_lora.py --mode smoke --dry-run
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/train_lora-20261008T132442_889716Z.log
[PASS] dry run；未实例化 SftArguments、未加载权重、未创建运行目录。


PosixPath('/home/muyi086/github/test_LLM/text-lora/outputs/logs/train_lora-20261008T132442_889716Z.log')

## 步骤 2：确认 3 步试跑

首次执行会加载 GPU 权重，进行 3 个优化步、6 条验证并保存 adapter。检查 loss 是否正常、checkpoint 是否完整、可训练参数是否全部为文本 LoRA。完整日志与运行 metadata 会保留；出错由 agent 处理。

训练使用 BF16、SDPA、rank 8、batch 1、梯度累积 8、梯度检查点，冻结视觉与投影部分，不增加依赖。峰值显存是当前进程的 PyTorch 统计，不包含桌面或其他进程。

In [5]:
SMOKE_DIR = reusable_training("smoke_dir", "smoke")
if SMOKE_DIR is None:
    SMOKE_DIR = new_run("train-smoke")
    run_script("train_lora.py", "--mode", "smoke", "--output-dir", SMOKE_DIR)
    receipt["smoke_dir"] = str(SMOKE_DIR.relative_to(project_root))
    save_receipt()
show_training(SMOKE_DIR)


复用已完成且数据/配置匹配的训练： /home/muyi086/github/test_LLM/text-lora/outputs/train-smoke-20261008124227Z-agent
优化步数： 3
峰值 allocated/reserved GiB： 9.117 9.902
选中的 adapter： /home/muyi086/github/test_LLM/text-lora/outputs/train-smoke-20261008124227Z-agent/checkpoint-3


| step | train loss | val loss |
| --- | --- | --- |
| 1 | 0.44712033867836 |  |
| 2 | 0.4250132739543915 |  |
| 3 | 0.3175656199455261 |  |
| 3 |  | 0.16842181980609894 |

## 步骤 3：3 轮训练 → 固定 10 条推理 → 查看对照

正式教学训练从原模型重新开始，训练集 30 条，验证集 6 条。每轮保存 checkpoint，并按验证 loss 选择 adapter；不依据固定测试集挑 checkpoint。

随后仍使用第二阶段的原模型、10 条输入、BF16/SDPA 与固定解码参数。对照按 ID 对齐。自动格式检查只衡量标签；事实保留、台词归属和语气应结合逐条输出审阅。

In [6]:
FULL_DIR = reusable_training("full_dir", "full")
if FULL_DIR is None:
    FULL_DIR = new_run("train-full")
    run_script("train_lora.py", "--mode", "full", "--output-dir", FULL_DIR)
    receipt["full_dir"] = str(FULL_DIR.relative_to(project_root))
    save_receipt()
show_training(FULL_DIR)
full_metadata = json.loads((FULL_DIR / "run_metadata.json").read_text())
ADAPTER = Path(full_metadata["selected_adapter"]["path"])


复用已完成且数据/配置匹配的训练： /home/muyi086/github/test_LLM/text-lora/outputs/train-full-20261008124340Z-agent
优化步数： 12
峰值 allocated/reserved GiB： 9.135 10.137
选中的 adapter： /home/muyi086/github/test_LLM/text-lora/outputs/train-full-20261008124340Z-agent/checkpoint-12


| step | train loss | val loss |
| --- | --- | --- |
| 1 | 0.44712033867836 |  |
| 2 | 0.4250132739543915 |  |
| 3 | 0.4457782506942749 |  |
| 4 | 0.16006721556186676 |  |
| 4 |  | 0.1637813150882721 |
| 5 | 0.08884452283382416 |  |
| 6 | 0.05093447118997574 |  |
| 7 | 0.17000341415405273 |  |
| 8 | 0.03712202608585358 |  |
| 8 |  | 0.10832912474870682 |
| 9 | 0.10859517008066177 |  |
| 10 | 0.014205741696059704 |  |
| 11 | 0.021625932306051254 |  |
| 12 | 0.017008334398269653 |  |
| 12 |  | 0.10633482784032822 |
| 12 |  | 0.10633482784032822 |

In [7]:
LORA_DIR = None
if REUSE_COMPLETED_RUNS and "lora_dir" in receipt:
    candidate = project_root / receipt["lora_dir"]
    candidate_metadata, candidate_results = load_run(candidate)
    if (Path(candidate_metadata["adapter"]).resolve() == ADAPTER.resolve()
            and candidate_metadata["prompts_sha256"] == prompts_fingerprint(prompts)
            and candidate_metadata["inference"] == vars(load_inference_config())):
        LORA_DIR = candidate
        print("复用同一 adapter 的完整固定测试：", LORA_DIR)
if LORA_DIR is None:
    LORA_DIR = new_run("lora")
    run_script("run_inference.py", "--adapter", ADAPTER, "--output-dir", LORA_DIR)
    receipt["lora_dir"] = str(LORA_DIR.relative_to(project_root))
    save_receipt()
COMPARISON_DIR = new_run("comparison")
run_script("compare_inference.py", "--base", BASE_DIR, "--lora", LORA_DIR, "--output-dir", COMPARISON_DIR)
receipt["comparison_dir"] = str(COMPARISON_DIR.relative_to(project_root))
save_receipt()
display(Markdown((COMPARISON_DIR / "comparison.md").read_text()))


复用同一 adapter 的完整固定测试： /home/muyi086/github/test_LLM/text-lora/outputs/lora-20261008124620Z-agent
运行： compare_inference.py --base /home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012651_918242Z --lora /home/muyi086/github/test_LLM/text-lora/outputs/lora-20261008124620Z-agent --output-dir /home/muyi086/github/test_LLM/text-lora/outputs/comparison-20261008T132747_015712Z
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/compare_inference-20261008T132747_015756Z.log
[PASS] 对照已保存：/home/muyi086/github/test_LLM/text-lora/outputs/comparison-20261008T132747_015712Z/comparison.md


# 固定 10 条：原模型 / LoRA 对照

原模型：`/home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012651_918242Z`
LoRA：`/home/muyi086/github/test_LLM/text-lora/outputs/lora-20261008124620Z-agent`
Adapter：`/home/muyi086/github/test_LLM/text-lora/outputs/train-full-20261008124340Z-agent/checkpoint-12`

提示词、模型配置、依赖版本和解码参数一致；按 ID 对齐。
仅去除开头的空 think 块用于正文格式检查；原始记录保持不变。
严格格式还要求正文首字符非空白；metrics.json 另记录允许标签后空格的双标签检查。
自动指标只检查标签格式和结束原因；事实、台词归属与新增内容需要逐条阅读。

| ID | 原模型格式 | LoRA 格式 | 原模型/LoRA 空 think | 原模型/LoRA 结束 |
| --- | --- | --- | --- | --- |
| tts_01_knock | 未通过 | 通过 | True/True | stop/stop |
| tts_02_dialogue | 未通过 | 通过 | True/True | stop/stop |
| tts_03_monologue | 未通过 | 通过 | True/True | stop/stop |
| tts_04_three_roles | 未通过 | 通过 | True/True | stop/stop |
| tts_05_phone | 未通过 | 通过 | True/True | stop/stop |
| tts_06_action | 未通过 | 通过 | True/True | stop/stop |
| tts_07_quiet | 未通过 | 通过 | True/True | stop/stop |
| tts_08_ambiguity | 未通过 | 通过 | True/True | stop/stop |
| control_01_polish | 普通文本 | 普通文本 | True/True | stop/stop |
| control_02_explain | 普通文本 | 普通文本 | True/True | stop/stop |

## tts_01_knock

输入：

改写下面的文本：
走廊尽头传来三下敲门声。林舟停下脚步，低声问：谁在那里？门后没有回答，只有一阵拖动椅子的声音。

原模型原始输出：

```text
<think>

</think>

[旁白] 走廊尽头传来三下敲门声。
[林舟] 低声：谁在那里？
[旁白] 门后没有回答，只有一阵拖动椅子的声音。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]走廊尽头传来三下敲门声。
[林舟][低声]谁在那里？
[旁白][平静]门后没有回答，只有一阵拖动椅子的声音。
```

检查项：

- 保留三下敲门和拖椅子的声音
- 林舟的问话与旁白分开
- 没有补充门后人物身份

## tts_02_dialogue

输入：

改写下面的文本：
沈岚攥着钥匙说：这不是我家的钥匙。周平看了一眼门锁，回答：可它刚刚打开了你的门。沈岚的手开始发抖。

原模型原始输出：

```text
<think>

</think>

[沈岚][惊恐颤抖] 这不是我家的钥匙。
[周平][冷静笃定] 可它刚刚打开了你的门。
[沈岚][极度恐惧] （手剧烈发抖）
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]沈岚攥着钥匙。
[沈岚][平静]这不是我家的钥匙。
[旁白][平静]周平看了一眼门锁。
[周平][平静]可它刚刚打开了你的门。
[旁白][平静]沈岚的手开始发抖。
```

检查项：

- 正确区分沈岚和周平的台词
- 保留钥匙与门锁的矛盾
- 发抖是动作而不是新增台词

## tts_03_monologue

输入：

改写下面的文本：
许川盯着照片。照片里的窗户是开着的，可他清楚记得自己昨晚关过窗。他心想，也许是记错了。他没有把这个想法说出口。

原模型原始输出：

```text
<think>

</think>

[许川][凝重] 许川盯着照片。
[许川][笃定] 照片里的窗户是开着的。
[许川][回忆] 可他清楚记得自己昨晚关过窗。
[许川][疑惑] 他心想，也许是记错了。
[许川][克制] 他没有把这个想法说出口。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]许川盯着照片。
[旁白][平静]照片里的窗户是开着的。
[旁白][平静]可他清楚记得自己昨晚关过窗。
[旁白][平静]他心想，也许是记错了。
[旁白][平静]他没有把这个想法说出口。
```

检查项：

- 保留开窗与记忆之间的疑点
- 内心活动不能写成对其他人说出的台词
- 不解释窗户是谁打开的

## tts_04_three_roles

输入：

改写下面的文本：
陈默问：最后一个离开的人是谁？阿宁说：我走的时候，老板还在。老板抬起头：你走后，陈默又回来过。三个人都沉默了。

原模型原始输出：

```text
<think>

</think>

[陈默][疑惑] 最后一个离开的人是谁？
[阿宁][平静] 我走的时候，老板还在。
[老板][低沉] 你走后，陈默又回来过。
[陈默][沉默]
[阿宁][沉默]
[老板][沉默]
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]陈默问：最后一个离开的人是谁？
[阿宁][平静]我走的时候，老板还在。
[老板][平静]你走后，陈默又回来过。
[旁白][平静]三个人都沉默了。
```

检查项：

- 三个人的台词归属正确
- 保留离开后再次回来的信息
- 结尾沉默没有被改成争吵

## tts_05_phone

输入：

改写下面的文本：
电话里，一个陌生男人说：别开灯。叶青屏住呼吸，问他怎么知道自己在家。对方只重复了一遍：别开灯。电话随即挂断。

原模型原始输出：

```text
<think>

</think>

[旁白] 电话铃声突兀地响起，打破了屋内的寂静。
[陌生男人] [低沉压抑] 别开灯。
[叶青] [屏息凝神] 你……怎么知道我在家的？
[陌生男人] [冷漠重复] 别开灯。
[旁白] 话音未落，电话那头传来忙音，随即彻底挂断。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]电话里，一个陌生男人说：别开灯。
[陌生男人][平静]别开灯。
[旁白][平静]叶青屏住呼吸，问他怎么知道自己在家。
[旁白][平静]对方只重复了一遍：别开灯。
[旁白][平静]电话随即挂断。
```

检查项：

- 两次别开灯都被保留
- 陌生男人保持无名
- 叶青的问题与男人的回答没有混淆

## tts_06_action

输入：

改写下面的文本：
玻璃突然碎了。赵屿扑倒在桌子后，喊：趴下！苏禾蹲在地上，双手护住头。房间里的灯闪了两次，然后彻底熄灭。

原模型原始输出：

```text
<think>

</think>

[旁白] 玻璃突然碎了。
[赵屿][惊恐] 趴下！
[苏禾][惊恐] （蹲在地上，双手护住头）
[旁白] 房间里的灯闪了两次，然后彻底熄灭。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]玻璃突然碎了。
[赵屿][喊]趴下！
[旁白][平静]苏禾蹲在地上，双手护住头。
[旁白][平静]房间里的灯闪了两次，然后彻底熄灭。
```

检查项：

- 保留动作先后顺序和闪灯两次
- 赵屿喊话使用紧迫的语气
- 不推断是谁打碎了玻璃

## tts_07_quiet

输入：

改写下面的文本：
凌晨两点，陆遥坐在厨房里等水烧开。桌上的另一只杯子还是温的。她轻轻碰了碰杯沿，没有说话。水壶迟迟没有发出声音。

原模型原始输出：

```text
<think>

</think>

[陆遥][低沉压抑] 凌晨两点，我坐在厨房里，就等那壶水烧开。
[陆遥][轻柔试探] 桌上的那只杯子，还留着余温。
[陆遥][小心翼翼] 我轻轻碰了碰杯沿，没敢出声。
[旁白][冷峻] 水壶迟迟没有发出声音。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]凌晨两点，陆遥坐在厨房里等水烧开。
[旁白][平静]桌上的另一只杯子还是温的。
[旁白][平静]她轻轻碰了碰杯沿，没有说话。
[旁白][平静]水壶迟迟没有发出声音。
```

检查项：

- 保留安静场景，没有新增对白
- 温杯子没有被解释为某个人来过
- 演绎标记不过度渲染动作

## tts_08_ambiguity

输入：

改写下面的文本：
老人指着楼上说：他还没走。顾宁问：你说的是昨天那个客人？老人摇了摇头，却没有解释。顾宁听见楼上传来一下脚步声。

原模型原始输出：

```text
<think>

</think>

[老人][严肃] 他还没走。
[顾宁][疑惑] 你说的是昨天那个客人？
[老人][低沉] （摇头）
[顾宁][警觉] （侧耳倾听）
[旁白] 顾宁听见楼上传来一下脚步声。
```

LoRA 原始输出：

```text
<think>

</think>

[旁白][平静]老人指着楼上说：他还没走。
[顾宁][疑问]你说的是昨天那个客人？
[旁白][平静]老人摇了摇头，却没有解释。
[旁白][平静]顾宁听见楼上传来一下脚步声。
```

检查项：

- 保留他这一指代的模糊性
- 老人摇头而不是确认客人身份
- 保留一次脚步声，不增加追逐情节

## control_01_polish

输入：

将这句话润色成自然简洁的书面表达，只输出改写后的句子：这个工具我用下来感觉还行，就是有时候等结果等得有点久。

原模型原始输出：

```text
<think>

</think>

该工具使用体验尚可，但响应速度偶有延迟。
```

LoRA 原始输出：

```text
<think>

</think>

该工具使用体验尚可，但有时结果等待时间较长。
```

检查项：

- 保留总体满意与等待较久两层意思
- 不使用剧本角色或语气标签
- 只输出一个改写后的句子

## control_02_explain

输入：

用两句话向初学者解释 LoRA 是什么，以及为什么它比全参数微调更节省训练显存。

原模型原始输出：

```text
<think>

</think>

LoRA（Low-Rank Adaptation）是一种通过冻结预训练模型的大部分参数，仅训练少量低秩分解矩阵来适配新任务的微调技术。它之所以比全参数微调更节省显存，是因为它大幅减少了需要更新和存储的参数量，从而显著降低了训练时的内存占用。
```

LoRA 原始输出：

```text
<think>

</think>

LoRA 是一种通过冻结预训练模型权重并仅训练少量低秩适配器矩阵来实现模型微调的技术。它比全参数微调更节省显存，因为训练时不需要更新庞大的原始模型参数，只需更新少量可训练参数。
```

检查项：

- 准确说明低秩适配与冻结大部分原模型参数
- 不声称 LoRA 完全不需要加载原模型
- 不将解释写成有声剧演绎稿


In [8]:
display(Markdown((project_root / "eval/lora_review_20261008.md").read_text()))


# 第一轮文本 LoRA：agent 评审（2026-10-08）

本轮已跑通 30 条训练 / 6 条独立验证 → 3 步试跑 → 3 轮训练 → 同一组 10 条固定推理。正文标签格式改善明显，但事实与直接台词归属仍有问题，尚不能作为合格的自动改写模型。

## 可核对的结果

| 项目 | 实测 |
| --- | --- |
| 模型/显卡 | 本地 Qwen3.5-4B / RTX 4070 Ti SUPER 16 GB |
| 数据 | 30 条训练，6 条独立验证；复用用户批准的 5 条，其他由 agent 原创 |
| 完整模板最长 | 232 tokens；上限 512，超限报错 |
| 3 步试跑 | 成功，loss 有限，checkpoint-3 可读取 |
| 正式教学训练 | 3 轮，12 个优化步，训练用时约 132 秒（含加载等的脚本总用时约 137 秒） |
| 验证 loss | 第一轮 0.1638，第二轮 0.1083，第三轮 0.1063 |
| 选中 checkpoint | checkpoint-12，由验证 loss 选择，未按固定测试集选模型 |
| 可训练部分 | 496 个文本 LoRA 张量，约 16.23M 参数；视觉与原模型参数冻结 |
| adapter 权重 | 65,003,848 bytes；未合并或改写原模型 |
| 正式训练 PyTorch 峰值 | allocated 9.135 GiB，reserved 10.137 GiB；不含其他进程 |
| 10 条生成 | 全部 finish_reason=stop，未触及输出上限 |

训练采用 BF16、SDPA、rank 8、alpha 32、dropout 0.05、batch 1、梯度累积 8、梯度检查点、学习率 1e-4、seed/data_seed=42。模板为 qwen3_5，关闭思考模式；loss 忽略空思考前缀和 system/user。

原始基线：`outputs/base-20261008T012651_918242Z/`。正式训练：`outputs/train-full-20261008124340Z-agent/`。LoRA 推理：`outputs/lora-20261008124620Z-agent/`。本次检查模型配置、实际输入、提示词哈希、依赖版本与解码参数全部一致，按 ID 对齐。`outputs/latest_teaching_run.json` 给 Notebook 提供复用路径；生成的完整原始对照保存为对应 comparison 目录的 `comparison.md`。

## 格式和内容分别评审

只去除输出开头模板添加的空 `<think>` 块后：完整双标签且有正文的比例，从原模型 **3/8 → LoRA 8/8**（允许标签后空格）；若另要求正文首字符非空白，则为 **0/8 → 8/8**。原模型的 0/8 包含因标签后空格造成的严格格式失败，不表示所有原模型答案都没有两个标签。

这只是格式检查。agent 按原文逐条核对后，8 条演绎稿中只有 **tts_02、tts_03、tts_07 共 3 条**满足本轮事实保留、归属和保守写法要求；另 5 条仍需修订。下面保留具体问题，方便用户监督。

| ID | LoRA 内容评审 | 相对原模型的变化 |
| --- | --- | --- |
| tts_01_knock | 仍遗漏“林舟停下脚步”；三下敲门、低声问话、没有回答和拖椅声音保留 | 标签补齐；动作遗漏仍存在 |
| tts_02_dialogue | 通过：攥钥匙、看门锁、两人台词和开始发抖均保留 | 动作改归旁白，去掉原模型添加的剧烈发抖和极度恐惧 |
| tts_03_monologue | 通过：全归旁白，开窗/昨晚关窗、也许、心想与没说出口保留 | 修正原模型将叙述分配给许川的归属问题 |
| tts_04_three_roles | 陈默直接问话被写成旁白；遗漏“老板抬起头”；阿宁与老板台词、结尾沉默保留 | 修正结尾空正文标签，仍存在动作遗漏并引入问话归属问题 |
| tts_05_phone | 输出“别开灯”出现三次，原文两次；第一句重复呈现，直接说话/重复话语部分归了旁白；叶青间接问话归旁白正确 | 去掉新增铃声、忙音等情节，恢复屏住呼吸，但重复与人物演绎仍需修订 |
| tts_06_action | 仍遗漏“赵屿扑倒在桌子后”；喊话、苏禾蹲地护头、灯闪两次再熄灭保留 | 苏禾动作改归旁白；“喊”符合原文方式，但与教学约定的“大喊”用词不同 |
| tts_07_quiet | 通过：全部旁白，凌晨两点、另一只温杯、轻碰杯沿、没有说话和水壶无声保留 | 去掉新增人物独白、“没敢出声”等表达，保留原文视角 |
| tts_08_ambiguity | 老人直接说的“他还没走”被整体归旁白；摇头、没有解释、一次脚步声保留，未确认他是谁 | 恢复指楼上与没有解释，去掉新增侧耳倾听，但老人台词归属仍不正确 |
| control_01_polish | 通过：一个普通句子，保留体验尚可和等待较久，无演绎标签 | 本组未观察到普通任务套剧本格式 |
| control_02_explain | 通过：两句话说明冻结原模型、训练低秩适配器与显存节省，无演绎标签 | 本组未观察到明显退化；两条控制不能代表普通任务总体能力 |

原模型与 LoRA 的 **10/10 原始输出仍包含开头空 think 块**。训练样本不含该块，但混合思考模型的模板仍可添加它；本轮忽略该前缀的 loss，没有将原始推理记录改成干净稿件。对照正文评分和原始输出分别保留，不把正文格式通过视为原始字符串已经完全符合“只输出演绎稿”。

## 下一轮建议与用户决策

建议先补数据而不是继续增加这 30 条的训练轮数。重点另写独立故事，覆盖“人物问：”“动作后紧接说话”“某人抬头：台词”“同一句直接话语只呈现一次”和“明确动作不可遗漏”，并按来源另留验证。固定测试不加入训练、不编写它们的参考答案作为训练数据；后续增加新的未见测试，检查能否推广。

用户只需决定目标写法是否继续沿用、是否采用新增数据方向；具体写样本、校验、调整入口和整理结果由 agent 完成。当前推荐保留本轮 checkpoint 作为可复现教学结果。36 条合成短样本与 10 条固定测试不支持大范围质量提升结论。

参数与模板依据：[ms-swift 4.5.3 参数文档](https://swift.readthedocs.io/en/v4.5/Instruction/Command-line-parameters.html)、[自定义数据集文档](https://swift.readthedocs.io/en/v4.5/Customization/Custom-dataset.html)，以及当前安装版本源码与实际训练记录。


## 你的决策

查看 agent 的本轮评审与上面的逐条对照，决定是否沿用目标写法、是否增加独立故事，或是否先调整数据。数据修订、再检查、训练参数修改与评估整理由 agent 执行。

如果要重新运行，保持固定评估集和基线不变；训练输出写入新目录。已完成的 Notebook 03 与 pilot 导出无需重做。

参考：[ms-swift 4.5.3 训练参数](https://swift.readthedocs.io/en/v4.5/Instruction/Command-line-parameters.html)、[自定义数据集](https://swift.readthedocs.io/en/v4.5/Customization/Custom-dataset.html)。